In [ ]:
%pip install -q scikit-learn==1.6.1
print("Installed scikit-learn 1.6.1. If Colab asks for a runtime restart, restart and run all cells.")

In [ ]:
import sklearn
print("scikit-learn:", sklearn.__version__)
assert sklearn.__version__ == "1.6.1", (
    "Use a Colab runtime with scikit-learn 1.6.1 for the required environment."
)

In [ ]:
import os

CSV_NAME = "UAAP_Season87_Womens_Volleyball_Full__1_.csv"

# Works in Google Colab and on a normal computer.
try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except ImportError:
    files = None
    IN_COLAB = False

if IN_COLAB and not os.path.exists(CSV_NAME):
    uploaded = files.upload()
    csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
    if not csv_files:
        raise FileNotFoundError("Upload the UAAP volleyball CSV file.")
    CSV_PATH = csv_files[0]
else:
    candidates = [CSV_NAME, os.path.join("..", CSV_NAME), os.path.join("data", CSV_NAME)]
    found = [p for p in candidates if os.path.exists(p)]
    if not found:
        raise FileNotFoundError(
            f"Could not find {CSV_NAME}. Put it next to this notebook or in the repo root."
        )
    CSV_PATH = found[0]
print("Using:", CSV_PATH)

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv(CSV_PATH)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())
display(df.tail())

assert len(df) == 120, f"Expected 120 records, found {len(df)}."
assert len(df) % 2 == 0, "Expected two adjacent rows per match."


In [ ]:
feature_columns = [
    "Team",
    "Opponent",
    "Round",
    "Team_Wins_Before",
    "Opp_Wins_Before",
    "Team_Win_Streak",
    "Opp_Win_Streak",
]
target_column = "Match_Outcome"

missing = [c for c in feature_columns + [target_column] if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

X = df[feature_columns].copy()
y = df[target_column].copy()

categorical_features = ["Team", "Opponent", "Round"]
numeric_features = [
    "Team_Wins_Before",
    "Opp_Wins_Before",
    "Team_Win_Streak",
    "Opp_Win_Streak",
]

split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split].copy(), X.iloc[split:].copy()
y_train, y_test = y.iloc[:split].copy(), y.iloc[split:].copy()

print(f"Training rows: {len(X_train)} (expected 96 rows / 48 matches)")
print(f"Testing rows:  {len(X_test)} (expected 24 rows / 12 matches)")
assert split == 96 and len(X_test) == 24
assert len(X_train) % 2 == 0 and len(X_test) % 2 == 0


In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

match_id = np.arange(len(X_train)) // 2
unique_matches = np.unique(match_id)

cv = [
    (
        np.where(np.isin(match_id, train_match_ids))[0],
        np.where(np.isin(match_id, valid_match_ids))[0],
    )
    for train_match_ids, valid_match_ids
    in TimeSeriesSplit(n_splits=5).split(unique_matches)
]

for fold_no, (train_idx, valid_idx) in enumerate(cv, start=1):
    assert len(train_idx) % 2 == 0 and len(valid_idx) % 2 == 0
    assert set(match_id[train_idx]).isdisjoint(set(match_id[valid_idx]))
    assert match_id[train_idx].max() < match_id[valid_idx].min()
    print(
        f"Fold {fold_no}: train rows={len(train_idx)}, "
        f"validation rows={len(valid_idx)}, "
        f"train matches={len(np.unique(match_id[train_idx]))}, "
        f"validation matches={len(np.unique(match_id[valid_idx]))}"
    )


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "K-Nearest Neighbors (K=5)": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(random_state=42),
}

pipelines = {
    name: Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator),
    ])
    for name, estimator in models.items()
}


In [ ]:
cv_results = []
for name, model in pipelines.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        n_jobs=None,
    )
    cv_results.append({
        "Model": name,
        "CV Mean Accuracy": scores.mean(),
        "CV Std": scores.std(),
        "CV Fold Scores": scores.tolist(),
    })

cv_df = pd.DataFrame(cv_results).sort_values(
    "CV Mean Accuracy", ascending=False
).reset_index(drop=True)

display(
    cv_df[["Model", "CV Mean Accuracy", "CV Std"]]
    .assign(**{
        "CV Mean Accuracy": lambda d: (d["CV Mean Accuracy"] * 100).round(2).astype(str) + "%",
        "CV Std": lambda d: (d["CV Std"] * 100).round(2).astype(str) + "%",
    })
)

winner_name = cv_df.loc[0, "Model"]
print("Winner by mean chronological CV accuracy:", winner_name)


In [ ]:
evaluation_rows = []
fitted_models = {}
test_predictions = {}

for name, model in pipelines.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    y_pred = model.predict(X_test)
    test_predictions[name] = y_pred

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    evaluation_rows.append({
        "Model": name,
        "CV Mean Accuracy": float(
            cv_df.loc[cv_df["Model"] == name, "CV Mean Accuracy"].iloc[0]
        ),
        "CV Std": float(cv_df.loc[cv_df["Model"] == name, "CV Std"].iloc[0]),
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Macro F1": macro_f1,
    })

results_df = pd.DataFrame(evaluation_rows).sort_values(
    "CV Mean Accuracy", ascending=False
).reset_index(drop=True)

formatted_results = results_df.copy()
for col in ["CV Mean Accuracy", "CV Std", "Test Accuracy", "Macro F1"]:
    formatted_results[col] = (formatted_results[col] * 100).round(2).astype(str) + "%"
display(formatted_results)

winner = fitted_models[winner_name]
winner_pred = test_predictions[winner_name]

print("Selected winner:", winner_name)
print("Test accuracy:", f"{accuracy_score(y_test, winner_pred) * 100:.2f}%")
print("\nConfusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, winner_pred, labels=[0, 1]))
print("\nClassification report:")
print(classification_report(y_test, winner_pred, labels=[0, 1], zero_division=0))


In [ ]:
import joblib
from pathlib import Path

MODEL_PATH = "uaap_volleyball_model.pkl"
joblib.dump(winner, MODEL_PATH)

loaded_model = joblib.load(MODEL_PATH)
assert isinstance(loaded_model, Pipeline)
assert np.array_equal(loaded_model.predict(X_test), winner_pred)

print("Saved and verified:", MODEL_PATH)
print("Pipeline type:", type(loaded_model).__name__)
print("scikit-learn version:", sklearn.__version__)

if IN_COLAB:
    files.download(MODEL_PATH)


In [ ]:
results_df.to_csv("uaap_chronological_model_results.csv", index=False)
print("Saved: uaap_chronological_model_results.csv")
if IN_COLAB:
    files.download("uaap_chronological_model_results.csv")
